# О bias и overscan

## Примеры bias изображений

Ниже представлены одиночный bias кадр и среднее из 100 bias кадров с камеры [Andor Apogee Aspen CG16M](http://www.andor.com/pdfs/specifications/Apogee_Aspen_CG16M_Specifications.pdf), недорогой CCD 4k × 4k с сенсором [Kodak KAF-16803](http://www.onsemi.com/pub/Collateral/KAF-16803-D.PDF). Эта модель камеры имеет типичный уровень bias около 1000 и шум чтения около 10 $e^-$, хотя точное значение варьируется от камеры к камере и зависит от температуры.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# Use custom style for larger fonts and figures
plt.style.use('guide.mplstyle')

In [ ]:
from astropy.nddata import CCDData
from astropy.visualization import hist

import numpy as np

from convenience_functions import show_image

In [ ]:
one_bias = CCDData.read('single_bias_thermoelectric.fit.bz2', unit='adu')
one_hundred_bias = CCDData.read('combined_bias_100_images.fit.bz2', unit='adu')

In [ ]:
fig, (ax_1_bias, ax_avg_bias) = plt.subplots(1, 2, figsize=(30, 15))

show_image(one_bias.data, cmap='gray', ax=ax_1_bias, fig=fig, input_ratio=8)
ax_1_bias.set_title('Single bias image')
show_image(one_hundred_bias.data, cmap='gray', ax=ax_avg_bias, fig=fig, input_ratio=8)
ax_avg_bias.set_title('100 bias images combined');

### Обратите внимание на несколько моментов

+ Уровень bias в этой конкретной камере составляет около 1023 (средний диапазон шкалы цветов).
+ Изображение ярче на левом и правом краях. Это "свечение усилителя" часто присутствует и вызвано электроникой CCD (фотодатчики с приложенным напряжением являются светодиодами).
+ Видны несколько вертикальных линий; это столбцы, для которых уровень bias постоянно выше.
+ Заметен "шум" в изображениях; это шум чтения.
+ Ни одно из изменений не является особенно большим.
+ Комбинирование нескольких bias изображений значительно снижает шум чтения. Этот пример немного нереалистичен тем, что было объединено 100 bias изображений, но он все же иллюстрирует идею о том, что комбинирование изображений уменьшает шум.

## Влияние комбинирования изображений на шум

Как подробно обсуждалось в [ноутбуке о комбинировании](01-06-Image-combination.ipynb), причина получения и комбинирования нескольких калибровочных изображений заключается в уменьшении шума, если изображения используются для калибровки. Разница между одиночным изображением и комбинацией изображений очевидна на изображениях выше. Другой способ увидеть влияние комбинирования изображений — это гистограмма значений пикселей. Обратите внимание, что распределение значений намного уже для объединенного изображения, чем для одиночного bias. Пиксели около краев, где велико свечение усилителя, группируются отдельно от остальных пикселей, чтобы подчеркнуть однородность чипа вдали от свечения.

In [ ]:
plt.figure(figsize=(20, 10))
hist(one_bias.data[:, 15:-30].flatten(), bins=800, alpha=0.4, label='One bias', color='deepskyblue')
hist(np.concatenate((one_bias.data[:, :15].flatten(), one_bias.data[:, -30:].flatten())), bins=400, alpha=0.2, label='One bias (edges only)', color='lightskyblue')
#hist(, bins=800, alpha=0.2, label='One bias (edges only)', color='darkblue')
hist(one_hundred_bias.data[:, 15:-30].flatten(), bins=800, alpha=0.4, label='One hundred bias images', color='darkgreen')
hist(np.concatenate((one_hundred_bias.data[:, :15].flatten(), one_hundred_bias.data[:, -30:].flatten())), bins=800, alpha=0.4, label='One hundred bias images (edges only)', color='lightgreen')

#hist(one_hundred_bias.data[:, :15].flatten(), bins=800, alpha=0.4, label='One hundred bias images', color='darkgreen')

plt.grid()
plt.xlim(975, 1400)
plt.legend()
plt.xlabel('Pixel value')
plt.ylabel('Number of pixels')
plt.semilogy();

## Обзор калибровки bias

Процесс здесь заключается в "калибровке" bias изображений путем вычитания overscan, если требуется, обрезке overscan из bias изображений, если он присутствует, и комбинировании всех bias изображений для создания "объединенного" bias (другие распространенные термины для этих изображений — "master" bias и иногда "super" bias).